In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from pathlib import Path

RESULTS = Path('../results')
DETECTORS = ['mahal', 'gmm']
DOMAINS   = ['synthetic', 'lightbox', 'sunlamp']
DETECTOR_LABELS = {'mahal': 'Mahalanobis', 'gmm': 'GMM'}
DOMAIN_COLORS   = {'synthetic': '#4C72B0', 'lightbox': '#DD8452', 'sunlamp': '#55A868'}

# look at the two pose-error components separately, not the blended SPEED score
COMPONENTS = {'E_R': 'rotation error  [deg]', 'E_T': 'translation error  [m]'}

def signed_log(scores):
    # from calibration.py
    return np.sign(scores) * np.log1p(np.abs(scores))

def load(detector, domain):
    scores = np.load(RESULTS / f'anomaly_scores_{detector}_{domain}.npy')
    df = pd.read_csv(RESULTS / f'per_image_errors_{domain}.csv')
    n = min(len(scores), len(df))
    errors = {comp: df[comp].values[:n] for comp in COMPONENTS}
    return signed_log(scores[:n]), errors


In [ ]:
# 2x3 figure (rows = detector, cols = domain) for a single error component
def plot_score_correlation(comp):
    comp_label = COMPONENTS[comp]
    fig, axes = plt.subplots(2, 3, figsize=(13, 8), sharey=False)
    fig.suptitle(f'Anomaly Score vs. {comp_label}\n(each point = one image)', fontsize=14)

    for row, det in enumerate(DETECTORS):
        for col, dom in enumerate(DOMAINS):
            ax = axes[row, col]
            scores, errors = load(det, dom)
            err = errors[comp]
            rho, pval = spearmanr(scores, err)
            ax.scatter(scores, err, s=6, alpha=0.35, color=DOMAIN_COLORS[dom], linewidths=0)
            ax.set_xlabel('Anomaly score  [signed log1p]', fontsize=10)
            ax.set_ylabel(comp_label, fontsize=10)
            ax.set_title(f'{DETECTOR_LABELS[det]} / {dom}', fontsize=11)
            sign = '+' if rho >= 0 else ''
            pstr = f'p={pval:.2e}' if pval < 0.001 else f'p={pval:.3f}'
            ax.text(0.97, 0.96, f'ρ = {sign}{rho:.3f}\n{pstr}',
                    transform=ax.transAxes, ha='right', va='top', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))

    plt.tight_layout()
    out = f'../figures/score_{comp}_correlation.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved → figures/score_{comp}_correlation.png')


# Rotation error
plot_score_correlation('E_R')


In [ ]:
# Translation error
plot_score_correlation('E_T')
